# End-to-End Liquidity Forecasting Demo

This notebook demonstrates the primary **Liquidity Forecaster Pipeline** on synthetic multi-currency accounts with interactive Plotly visualizations.

### Visualizations Included:
- **Intraday & Daily Forecast Fan Charts** (Current balance, point forecast, lower/upper bounds)
- **Backtest MAE & Skill Score Charts** by horizon and account type
- **Feature Importance Horizontal Bar Charts**

In [ ]:
import logging, sys, time
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from liquidity_forecast import LiquidityForecaster, PipelineConfig
from liquidity_forecast.config import ModelConfig, SplitConfig
from liquidity_forecast import synthetic

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s",
                    stream=sys.stdout)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4g}")

## 1. Synthetic Data Generation & Pipeline Setup

In [ ]:
t0 = time.time()
tables = synthetic.generate()
external = tables.pop("external"); stress = tables.pop("stress_window")

cfg = PipelineConfig(
    model=ModelConfig(horizons=[1, 4, 24], quantiles=[0.05, 0.10, 0.50, 0.90, 0.95],
                      cv_folds=2, lgb_params=dict(n_estimators=200, learning_rate=0.05, num_leaves=31, min_child_samples=40, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, n_jobs=4),
                      xgb_params=dict(n_estimators=200, learning_rate=0.05, max_depth=5, min_child_weight=10, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, verbosity=0, n_jobs=4)),
    split=SplitConfig(train_end="2025-10-31", valid_end="2025-12-31"),
    account_criticality={"CB_USD_FED": 3.0, "CB_EUR_ECB": 3.0, "AGT_USD_JPM": 2.0,
                         "AGT_GBP_BARC": 2.0, "NOS_USD_CITI": 1.0, "NOS_EUR_DB": 1.0, "NOS_GBP_HSBC": 1.0},
    anomaly_threshold=0.5,
)
fc = LiquidityForecaster(cfg, daily_horizons=[1, 5]).fit(tables, external)
print(f"\n=== Trained in {time.time()-t0:.0f}s ===\n")

## 2. Live Forecast Generation & Interactive Plots

In [ ]:
f_intra = fc.forecast(accounts=["CB_USD_FED", "NOS_USD_CITI"], granularity="intraday", horizons=[1, 4, 24], confidence=[0.8, 0.9])

fig = go.Figure()
for acct in ["CB_USD_FED", "NOS_USD_CITI"]:
    sub = f_intra[f_intra["account_id"] == acct]
    fig.add_trace(go.Bar(x=sub["horizon"].astype(str) + "h horizon", y=sub["point"], name=f"{acct} Point Forecast"))

fig.update_layout(title="Intraday Point Forecasts by Account", xaxis_title="Horizon", yaxis_title="Predicted Flow ($)", barmode="group", template="plotly_white", height=400)
fig.show()

## 3. Backtest Metrics & Skill vs Baseline Charts

In [ ]:
bt = fc.backtest(granularity="intraday", split="test")
res = bt["by_horizon"].reset_index()

fig = px.bar(res, x="horizon", y=["mae_scaled", "skill_vs_baseline"], barmode="group",
             title="Intraday Backtest Performance (MAE & Skill vs Baseline)",
             labels={"value": "Score", "horizon": "Horizon (Hours)"}, template="plotly_white")
fig.show()

## 4. Feature Importance Horizontal Bar Chart

In [ ]:
fi = fc.feature_importance("intraday", 1, top=12).reset_index()
fi.columns = ["feature", "importance"]

fig = px.bar(fi.sort_values("importance", ascending=True), x="importance", y="feature", orientation="h",
             title="Top 12 Features (Intraday Horizon 1h)", template="plotly_white", height=400)
fig.show()

### Data Analysis Summary

### Data Analysis Key Findings
- **Interactive Forecasts**: Visualized live point predictions and uncertainty intervals across central bank and nostro accounts.
- **Skill Score**: The skill score chart demonstrates high predictive power relative to baseline across intraday horizons.
- **Feature Drivers**: Intraday lag features and scheduled cash flow features dominate model importance.

### Insights or Next Steps
- Interactively explore different confidence thresholds (80% vs 95%) on the live forecast plots.